In [22]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import sys
sys.path.append("/Users/abhijeetpokhriyal/code/bringalive")

In [3]:
from bringalive.constants import PATHS, VECTORDB

In [4]:
import chromadb


ModuleNotFoundError: No module named 'chromadb'

In [4]:
game_collection = "game_memory"

In [5]:
client = chromadb.PersistentClient(path=PATHS.path_to_vectordb.value)
try:
    collection = client.delete_collection(game_collection)
except Exception as e:
    print(e)

In [4]:
# from bringalive.business_logic.agent import RAGAgent , Agent
from bringalive.business_logic.file_handlers.text_handler import TextHandler
from bringalive.constants import MODELS
# from bringalive.business_logic.memory import Memory

In [5]:
system_prompt = """
stick to what is asked. 
extract key phrases from the text that include names of people, places, things, events , dates.
stitch them together into a coherent summary adding fillers where needed.
output should only include the stitched together summary
"""

In [6]:
chunk_size = 2048
overlap_size = chunk_size//300
chunk_size, overlap_size

(2048, 6)

In [7]:
path_to_file = "/Users/abhijeetpokhriyal/code/bringalive/bringalive/documents/gandhi_experiments_with_truth.txt"

In [8]:
handler = TextHandler(path_to_file, chunk_size, overlap_size)

In [9]:
chunks = handler.load().transform().get()

In [10]:
current_state = chunks[20].content
current_state[:20]

'nd in those days the'

In [12]:
import requests
import json
def get_model_response(model,system_prompt,prompt,stream,think,options={} ,endpoint="http://localhost:11434/api/chat"):
    messages = []
    messages.append({
        "role": "system", "content": system_prompt
    })
    messages.append({
        "role": "user",
        "content": prompt
    })
    payload = {"model": model, "messages": messages,
                "think": think, "stream": stream, "options": options}

    response = requests.post(endpoint, json=payload, stream=stream)
    for line in response.iter_lines():
        if line:
            decoded_line = line.decode('utf-8')
            yield json.loads(decoded_line)["message"]


In [53]:
for m in get_model_response("llama3.2:1b",system_prompt , chunks[0].content , True,False):
    print(m)

{'role': 'assistant', 'content': 'The'}
{'role': 'assistant', 'content': ' Story'}
{'role': 'assistant', 'content': ' Of'}
{'role': 'assistant', 'content': ' My'}
{'role': 'assistant', 'content': ' Ex'}
{'role': 'assistant', 'content': 'periments'}
{'role': 'assistant', 'content': ' With'}
{'role': 'assistant', 'content': ' Truth'}
{'role': 'assistant', 'content': ' is'}
{'role': 'assistant', 'content': ' a'}
{'role': 'assistant', 'content': ' book'}
{'role': 'assistant', 'content': ' written'}
{'role': 'assistant', 'content': ' by'}
{'role': 'assistant', 'content': ' Mah'}
{'role': 'assistant', 'content': 'at'}
{'role': 'assistant', 'content': 'ma'}
{'role': 'assistant', 'content': ' Gandhi'}
{'role': 'assistant', 'content': ','}
{'role': 'assistant', 'content': ' first'}
{'role': 'assistant', 'content': ' published'}
{'role': 'assistant', 'content': ' in'}
{'role': 'assistant', 'content': ' Gujar'}
{'role': 'assistant', 'content': 'ati'}
{'role': 'assistant', 'content': ' in'}
{'role

In [13]:
all_results = []
with open("./output.txt" , "w") as f:
    for chunk_i in range(0,2):
        current_state = chunks[chunk_i].content
        # resp = agent.invoke(current_state)
        resp  =get_model_response(MODELS.default_model.value,system_prompt , current_state, False,False)
        msg = []
        for m in resp:
            msg.append(m)
        model_thought = "".join([m["thinking"]
                                for m in msg if m.get("thinking", None) is not None])
        # current_state = model_thought
        # memory.add_memories_from_text(model_thought, chunk_size, overlap_size)
        model_resp = "".join([m["content"]
                            for m in msg if m.get("thinking", None) == None])
        all_results.append((model_thought , model_resp))
        f.write(model_resp)

In [5]:
text = open("./output.txt" , "r").read()
text[:20]

'Here is the stitched'

In [7]:
# import torch
# from TTS.tts.configs.xtts_config import XttsConfig

# torch.serialization.add_safe_globals([XttsConfig])

In [2]:
from TTS.api import TTS

# TTS().list_models().list_models()

/Users/abhijeetpokhriyal/miniforge3/envs/llm/lib/python3.10/site-packages/librosa/core/intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [ ]:
# ensure_transformers_exports()
# tts = TTS(model_name="tts_models/multilingual/multi-dataset/your_tts", gpu=False)
tts = TTS(model_name="tts_models/multilingual/multi-dataset/xtts_v2", gpu=False)

TypeError: TTS.__init__() got an unexpected keyword argument 'use_deepspeed'

In [ ]:
# Faster XTTS generation on Apple Silicon / CPU
import re
import torch
from pathlib import Path

# Use MPS if available
use_gpu = False
#torch.backends.mps.is_available()
# print('MPS available:', use_gpu)

# Tune CPU threads (adjust 4/6/8 if needed)
torch.set_num_threads(20)

speaker_ref = "/Users/abhijeetpokhriyal/Downloads/voice_clone.wav"

tts.tts_to_file(
    text=text,
    file_path="./output4.wav",
    language="en",
    speed=0.9,
    speaker_wav=["/Users/abhijeetpokhriyal/Downloads/voice_clone.wav",
                 "/Users/abhijeetpokhriyal/Downloads/voice_clone_2.wav"]
)

 > Text splitted to sentences.
['Here is the stitched together summary:', 'Mahatma Gandhi was born to Mohanlal and Putlibai, a devout Hindu family in Porbandar, India.', 'He grew up with his elder brother, Harilal, and spent his childhood playing and learning from local priests.', 'At the age of 13, he married Kasturba, who was nine years his senior.', "The couple's marriage was marked by tension, particularly when Gandhi's father died, leaving him feeling guilty.", 'As a student at the high school in Rajkot, Gandhi began to explore spirituality and philosophical ideas.', 'A tragic event during his teenage years further shaped his perspective on truth.', 'He became an outcaste after being arrested for stealing, which he later atoned for through self-imposed punishments.', "Gandhi's desire for spiritual growth led him to England in 1893, where he was exposed to Western philosophy and ideas.", 'He struggled with feelings of shyness, which he called his "shield," but ultimately developed 

'./output3.wav'